# CIFAR-10 com CNN — Relatório de experimentos (grid search em blocos)

**Metodologia.** Em vez de escolher um valor por vez, cada bloco executa um **grid search** e o vencedor é
decidido pelos dados. O campeão de um bloco é lido automaticamente pelo bloco seguinte, sem nenhum valor escolhido à mão:

| Bloco | Pergunta | Grid |
|---|---|---|
| 0 — Referência | Onde partimos? | LeNet-5 do notebook original |
| 1 — Topologia | Qual estrutura base absorve melhor o CIFAR-10? | blocos × kernel × padding × redução (pool/stride) |
| 2 — Otimização | Qual algoritmo e taxa de aprendizagem fazem a rede campeã convergir melhor? | otimizador × lr |
| Checagem | A escolha em blocos se sustenta? | 2º e 3º do Bloco 1 com o otimizador campeão |
| 3 — Pooling e regularização | Como a janela de pooling, dropout e batch norm afetam a generalização? | pool × dropout × batch norm |
| Complementar B — Pooling | Qual janela de pooling é melhor, e como ela interage com a profundidade? | blocos × janela de pooling |
| Bônus — Data augmentation | Quanto um pré-processamento (crop + flip) acrescenta à rede campeã? | augmentation × blocos × dropout |

O notebook executa **o estudo completo da CNN em uma única execução** (Save & Run All): cada bloco herda o campeão do
anterior, e os complementares e o bônus partem do campeão mais recente.

**Protocolo de avaliação.**
- **Validação cruzada estratificada em 5 folds** sobre as 50.000 imagens de treino, com as mesmas partições em todos os experimentos (comparações pareadas).
- **Triagem:** cada configuração roda os folds 1–3; as 3 melhores **completam os folds 4–5** (sem refazer nada) e o campeão é a maior média nos 5 folds.
- **Escolha sempre pela validação.** O conjunto de teste (10.000 imagens) só é revelado na seção final, para os campeões. Usá-lo para escolher tornaria o número final otimista (vazamento de dados).
- Diferenças menores que o desvio padrão entre folds não devem ser tratadas como melhora real.

**Registro dos resultados.** Cada experimento grava em `/kaggle/working/outputs/{exp_name}/`:

| Arquivo | Conteúdo |
|---|---|
| `parametros.json` | hiperparâmetros exatos, arquitetura resolvida, versões e commit |
| `historico_treino.csv` | uma linha por **fold × época**: loss, acurácia, precision/recall/F1 (gerais e por classe) de treino e validação, gap |
| `historico_treino_agregado.csv` | média e desvio padrão entre folds, por época |
| `resultados.json` | resultado de cada fold e médias (validação na melhor época e teste) |
| `melhor_modelo.pth` | pesos com a menor `val/loss` (melhor fold); cada fold em `folds/fold_k/` |

Cada bloco grava em `outputs/_grids/{bloco}/`: `configs.csv`, `ranking_triagem.csv`, `ranking_final.csv`, `campeao.json` e `logs/`.
Tabelas e figuras dos relatórios ficam em `outputs/_relatorio/`. Tudo é gravado a cada época, antes do envio ao Weights & Biases.

**Tempo estimado:** ~6 h com 1 GPU T4 ou **~3h–3h30 com 2× T4** (estimativa a partir de medições; varia ±30%).
Os experimentos são distribuídos entre as GPUs visíveis (1 por GPU, ajustável em `WORKERS_PER_GPU`).

**Antes de executar (Kaggle):**
1. *Settings → Accelerator*: **GPU T4 ×2**. *Settings → Internet*: **On**.
   *Add Input → Datasets*: busque **cifar10-python** e anexe um dataset com a versão Python do CIFAR-10
   (pasta `cifar-10-batches-py` ou arquivo `cifar-10-python.tar.gz`). Sem ele, o download do servidor original é muito lento.
2. *Add-ons → Secrets*: `GITHUB_TOKEN` (obrigatório se o repositório for privado) e `WANDB_API_KEY`
   (opcional — sem ele tudo continua salvo localmente). Marque os dois como anexados a este notebook.
3. Use **Save Version → Save & Run All (Commit)**: `/kaggle/working` (com `outputs/` e `outputs.zip`) fica salvo na aba *Output*.
4. **Retomada:** tudo é retomável. Se a sessão cair, adicione o Output da versão anterior como *Input*, preencha `RESUME_FROM`
   e rode de novo: experimentos e folds concluídos são pulados.

## 0. Preparação do ambiente

In [ ]:
import os

REPO_URL = "https://github.com/diegoflyra/dfal-neural-networks.git"
REPO_DIR = "/tmp/dfal-neural-networks"  # fora de /kaggle/working: código e dataset não poluem o Output

# Repositório privado: crie o secret GITHUB_TOKEN (Add-ons → Secrets) com um token de leitura do GitHub.
# O token fica só em /tmp (não vai para o Output) e nunca é impresso.
clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    clone_url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    print("GitHub: usando GITHUB_TOKEN")
except Exception:
    print("GitHub: sem GITHUB_TOKEN (funciona apenas se o repositório for público)")

if os.path.isdir(REPO_DIR):
    !git -C $REPO_DIR pull -q
else:
    !git clone -q $clone_url $REPO_DIR
%cd $REPO_DIR
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import shutil
import sys

import pandas as pd

WORKERS_PER_GPU = 1  # experimentos simultâneos por GPU
RESUME_FROM = ""  # ex.: "/kaggle/input/<output-da-versao-anterior>/outputs" para retomar

# CIFAR-10: usa a cópia anexada como Input do Kaggle (segundos), em vez do servidor original (lento).
# A cópia só é aceita se TODOS os arquivos tiverem o MD5 oficial (os mesmos hashes que o torchvision
# usa para validar o download de https://www.cs.toronto.edu/~kriz/cifar.html). Se algo divergir,
# a cópia é descartada e o dataset é baixado do servidor original.
import glob
import hashlib
import tarfile

from torchvision.datasets import CIFAR10

DATA_DIR = os.path.join(REPO_DIR, "data")
CIFAR_DIR = os.path.join(DATA_DIR, "cifar-10-batches-py")
OFFICIAL_MD5 = dict(CIFAR10.train_list + CIFAR10.test_list + [[CIFAR10.meta["filename"], CIFAR10.meta["md5"]]])
os.makedirs(DATA_DIR, exist_ok=True)


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def cifar_is_official(folder):
    rows, ok = [], True
    for name, expected in OFFICIAL_MD5.items():
        path = os.path.join(folder, name)
        actual = md5(path) if os.path.isfile(path) else "arquivo ausente"
        ok &= actual == expected
        rows.append({"arquivo": name, "md5 oficial": expected, "md5 da cópia": actual,
                     "status": "OK" if actual == expected else "DIFERENTE"})
    display(pd.DataFrame(rows))
    return ok


if not os.path.isdir(CIFAR_DIR):
    folders = glob.glob("/kaggle/input/**/cifar-10-batches-py", recursive=True)
    archives = glob.glob("/kaggle/input/**/cifar-10-python.tar.gz", recursive=True)
    if folders:
        print(f"Cópia encontrada: {folders[0]}")
        shutil.copytree(folders[0], CIFAR_DIR)
    elif archives:
        print(f"Arquivo encontrado: {archives[0]} | md5 {md5(archives[0])} (oficial: {CIFAR10.tgz_md5})")
        with tarfile.open(archives[0]) as tar:
            tar.extractall(DATA_DIR)
    else:
        print("AVISO: CIFAR-10 não encontrado nos Inputs; será baixado do servidor original (pode levar muitos minutos).")

if os.path.isdir(CIFAR_DIR):
    if cifar_is_official(CIFAR_DIR):
        print("CIFAR-10 verificado: todos os arquivos são idênticos aos oficiais.")
    else:
        shutil.rmtree(CIFAR_DIR)
        print("CÓPIA REJEITADA: arquivos diferentes dos oficiais. O dataset será baixado do servidor original.")

# Onde os resultados são gravados (lido por run_experiment.py, grid_search.py e report_utils.py)
os.environ["EXP_OUTPUT_DIR"] = "/kaggle/working/outputs"
os.makedirs(os.environ["EXP_OUTPUT_DIR"], exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, os.environ["EXP_OUTPUT_DIR"], dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

# Weights & Biases via Kaggle Secrets; se falhar, os experimentos seguem apenas com o registro local.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: chave carregada.")
except Exception as exc:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado ({exc}). Resultados continuam em {os.environ['EXP_OUTPUT_DIR']}.")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import report_utils as rep

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino):
# GPUs, flags → arquitetura, carregamento em GPU/K-fold (baixa o CIFAR-10) e construção de todas as configurações dos grids
!nvidia-smi -L
!python tests/check_hyperparams.py
!python tests/check_data_loader.py
!python tests/check_grids.py

## Bloco 0 — Referência

Mesmo protocolo dos blocos seguintes (Adam, lr=1e-3, batch 128, early stopping com paciência 5, 5 folds), aplicado à
LeNet-5 adaptada do notebook original.

Fixos no bloco: `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=40`, `patience=5`, `eval_train=True`, `activation=relu`, `loss_fn=cross_entropy`, `filters=64`, `filters_growth=double`, `convs_per_block=1`, `fc_neurons=[512]`, `cnn_dropout=0.0`. Todas as configurações rodam os 5 folds.

In [ ]:
!python src/grid_search.py grids/cnn_b0_referencia.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.grid_ranking("cnn_b0_referencia", "final")

## Bloco 1 — Topologia e extração espacial

**Pergunta:** qual estrutura convolucional absorve melhor o CIFAR-10: quantos blocos, qual janela de convolução,
qual padding e como reduzir a resolução espacial?

Cada bloco é `Conv2d → Ativação → redução`, com 64 filtros no primeiro bloco, dobrando a cada bloco.
A redução de resolução é tratada como **uma** variável — max pooling 2×2 (stride 1) **ou** convolução com stride 2
(sem pooling) —, pois combinar os dois reduz a imagem 4× por bloco e colapsa redes profundas. Combinações cujo
mapa de ativação chega a 0×0 são descartadas automaticamente antes do treino.
Sem regularização e com otimização padrão; early stopping na `val/loss`.

| Eixo | Valores |
|---|---|
| `conv_blocks` | `2`, `3`, `4` |
| `kernel_size` | `3`, `5` |
| `padding` | `same`, `valid` |
| `reducao` | **maxpool** (`stride=1`, `pool_size=2`), **stride2** (`stride=2`, `pool_size=0`) |

**24 combinações.** Fixos no bloco: `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=40`, `patience=5`, `eval_train=True`, `activation=relu`, `loss_fn=cross_entropy`, `filters=64`, `filters_growth=double`, `convs_per_block=1`, `fc_neurons=[512]`, `cnn_dropout=0.0`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds. Limite: 20,000,000 parâmetros.

**O que observar:** efeito da profundidade, kernel 3×3 vs 5×5 (campo receptivo × parâmetros), `same` vs `valid`
(perda de bordas) e max pooling vs stride (invariância local × redução aprendida).

In [ ]:
!python src/grid_search.py grids/cnn_b1_topologia.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `cnn_b1_topologia`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("cnn_b1_topologia", "triagem")

In [ ]:
rep.discarded_configs("cnn_b1_topologia")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.heatmap("cnn_b1_topologia", row="conv_blocks", col=["kernel_size", "padding"], facet="reducao")

In [ ]:
rep.plot_grid_bars("cnn_b1_topologia")

#### Confirmação das finalistas e campeão — `cnn_b1_topologia`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("cnn_b1_topologia", "final")

In [ ]:
rep.show_champion("cnn_b1_topologia")
rep.plot_finalists("cnn_b1_topologia")

In [ ]:
rep.heatmap("cnn_b1_topologia", row="conv_blocks", col=["kernel_size", "padding"], facet="reducao", value="num_parameters")

### 📝 Análise — Bloco 1

- **Profundidade: quantos blocos foram úteis?** _…_
- **Kernel 3×3 vs 5×5:** _…_
- **Padding same vs valid:** _…_
- **Max pooling vs stride 2:** _…_
- **Diferença do campeão para a referência (validação):** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 2 — Otimização

**Pergunta:** com a topologia campeã do Bloco 1 fixa, qual combinação de algoritmo e taxa de aprendizagem converge melhor?

**Base:** o campeão do Bloco 1 (abaixo). SGD usa momentum 0,9. O grid de lr é comum aos dois algoritmos de propósito,
para mostrar a faixa útil de cada um.

| Eixo | Valores |
|---|---|
| `optimizer` | `sgd`, `adam` |
| `lr` | `0.0001`, `0.0003`, `0.001`, `0.003`, `0.01`, `0.03`, `0.1` |

**14 combinações.** Herdado do campeão de: `cnn_b1_topologia`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds.

In [ ]:
rep.show_champion("cnn_b1_topologia")

In [ ]:
!python src/grid_search.py grids/cnn_b2_otimizacao.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `cnn_b2_otimizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("cnn_b2_otimizacao", "triagem")

In [ ]:
rep.heatmap("cnn_b2_otimizacao", row="optimizer", col="lr")

In [ ]:
rep.plot_grid_bars("cnn_b2_otimizacao")

#### Confirmação das finalistas e campeão — `cnn_b2_otimizacao`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("cnn_b2_otimizacao", "final")

In [ ]:
rep.show_champion("cnn_b2_otimizacao")
rep.plot_finalists("cnn_b2_otimizacao")

In [ ]:
rep.heatmap("cnn_b2_otimizacao", row="optimizer", col="lr", value="melhor_epoca_media")

### 📝 Análise — Bloco 2

- **Faixa de lr útil para SGD e para Adam:** _…_
- **Qual convergiu em menos épocas?** _…_
- **Houve divergência?** _…_
- **Ganho sobre o Bloco 1 (validação):** _…_

### Checagem de interação

O 2º e o 3º colocados do Bloco 1 são treinados com o otimizador/lr campeões do Bloco 2 (5 folds).
O Bloco 3 parte da melhor rede entre o campeão do Bloco 2 e esta checagem.

Herdado do campeão de: `cnn_b2_otimizacao`. Todas as configurações rodam os 5 folds.

In [ ]:
!python src/grid_search.py grids/cnn_b2_checagem.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
pd.concat([rep.grid_ranking("cnn_b2_otimizacao", "final").head(1),
           rep.grid_ranking("cnn_b2_checagem", "final")], ignore_index=True)

### 📝 Análise — Checagem

- **A ordem das topologias se manteve com o novo otimizador?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bloco 3 — Janela de pooling e regularização

**Pergunta:** com a rede convergindo bem, como a janela de max pooling, o dropout nas camadas densas e a
batch normalization (após cada convolução) afetam a generalização?

**Base:** a melhor rede entre o campeão do Bloco 2 e a checagem. `pool_size=0` remove o pooling; combinações que
colapsam o mapa espacial ou excedem o limite de parâmetros (ex.: sem pooling em uma rede que dependia dele) são
descartadas e listadas abaixo. Épocas e paciência aumentadas.

| Eixo | Valores |
|---|---|
| `pool_size` | `0`, `2`, `3` |
| `cnn_dropout` | `0.0`, `0.3`, `0.5` |
| `cnn_batch_norm` | `False`, `True` |

**18 combinações.** Herdado do campeão de: `cnn_b2_otimizacao`, `cnn_b2_checagem`. Fixos no bloco: `epochs=50`, `patience=7`. Triagem nos folds [1, 2, 3] de 5; as 3 melhores completam os 5 folds. Limite: 20,000,000 parâmetros.

**O que observar:** o `gap/accuracy` com dropout e batch norm, e o efeito de pooling mais agressivo (3×3).

In [ ]:
!python src/grid_search.py grids/cnn_b3_regularizacao.json --workers_per_gpu $WORKERS_PER_GPU

#### Resultados da triagem — `cnn_b3_regularizacao`

Todas as configurações, ordenadas pela **validação** (média ± desvio nos folds de triagem). As métricas de teste não aparecem aqui de propósito.

In [ ]:
rep.grid_ranking("cnn_b3_regularizacao", "triagem")

In [ ]:
rep.discarded_configs("cnn_b3_regularizacao")  # combinações descartadas antes do treino e o motivo

In [ ]:
rep.heatmap("cnn_b3_regularizacao", row="cnn_dropout", col="cnn_batch_norm", facet="pool_size")

In [ ]:
rep.plot_grid_bars("cnn_b3_regularizacao")

#### Confirmação das finalistas e campeão — `cnn_b3_regularizacao`

As finalistas completaram todos os folds; o campeão é a maior `val/accuracy` média nos K folds.

In [ ]:
rep.grid_ranking("cnn_b3_regularizacao", "final")

In [ ]:
rep.show_champion("cnn_b3_regularizacao")
rep.plot_finalists("cnn_b3_regularizacao")

In [ ]:
rep.heatmap("cnn_b3_regularizacao", row="cnn_dropout", col="cnn_batch_norm", facet="pool_size", value="gap/accuracy_mean")

### 📝 Análise — Bloco 3

- **Pooling 2×2 vs 3×3 (ou sem pooling):** _…_
- **Dropout e batch norm reduziram o gap?** _…_
- **Houve subajuste com regularização forte?** _…_
- **Ganho sobre o Bloco 2 (validação):** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Complementar B — Janela de pooling

**Pergunta:** qual janela de max pooling extrai melhor as características espaciais, e como isso interage com a
profundidade? Janelas maiores reduzem a resolução mais rápido (menos parâmetros), mas descartam informação espacial mais cedo.
No Bloco 3, com a profundidade do campeão, várias janelas colapsavam o mapa espacial; aqui a profundidade também varia.

**Base:** o campeão do Bloco 3 (BatchNorm, dropout, otimizador). A combinação idêntica ao campeão é re-treinada e serve de
**checagem de reprodutibilidade**.

| Eixo | Valores |
|---|---|
| `conv_blocks` | `2`, `3`, `4` |
| `pool_size` | `2`, `3`, `4` |

**9 combinações.** Herdado do campeão de: `cnn_b3_regularizacao`. Todas as configurações rodam os 5 folds. Limite: 20,000,000 parâmetros.

**O que observar:** o heatmap blocos × pooling, o tamanho final do mapa espacial (`cnn_shape_saida_por_bloco` em
`parametros.json`), o número de parâmetros e se a combinação igual ao campeão reproduz a validação do Bloco 3.

In [ ]:
!python src/grid_search.py grids/cnn_b3b_pooling.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.discarded_configs("cnn_b3b_pooling")

In [ ]:
rep.grid_ranking("cnn_b3b_pooling", "final")

In [ ]:
rep.heatmap("cnn_b3b_pooling", row="conv_blocks", col="pool_size")

In [ ]:
rep.heatmap("cnn_b3b_pooling", row="conv_blocks", col="pool_size", value="num_parameters")

In [ ]:
rep.heatmap("cnn_b3b_pooling", row="conv_blocks", col="pool_size", value="gap/accuracy_mean")

In [ ]:
rep.show_champion("cnn_b3b_pooling")
rep.plot_finalists("cnn_b3b_pooling")

#### Comparação pareada e reprodutibilidade

Referência: a combinação idêntica ao campeão do Bloco 3. A tabela seguinte compara a validação dessa repetição com a execução original do Bloco 3 (mesmos folds e seed).

In [ ]:
rep.paired_comparison(rep.base_config_name("cnn_b3b_pooling"), "cnn_b3b_pooling__*")

In [ ]:
repeticao = rep.base_config_name("cnn_b3b_pooling")
pd.concat([rep.grid_ranking("cnn_b3_regularizacao", "final").head(1).assign(execucao="Bloco 3 (original)"),
           rep.grid_ranking("cnn_b3b_pooling", "final").query("exp_name == @repeticao").assign(execucao="Complementar B (repetição)")],
          ignore_index=True)[["execucao", "exp_name", "val/accuracy_mean", "val/accuracy_std", "melhor_epoca_media"]]

### 📝 Análise — Complementar B

- **Qual janela de pooling foi melhor? Depende da profundidade?** _…_
- **Relação entre tamanho final do mapa, parâmetros e desempenho:** _…_
- **A repetição reproduziu o campeão do Bloco 3? Qual o ruído entre execuções?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Bônus — Data augmentation

Data augmentation é um **pré-processamento**, não um hiperparâmetro da rede: por isso fica fora da sequência principal
de blocos. Cria variações plausíveis de cada imagem de treino a cada época, aumentando a diversidade efetiva dos dados.

**Transformações** (apenas no treino, direto na GPU; validação e teste usam as imagens originais):
**RandomCrop 32 com padding 4** (desloca até 4 pixels) e **RandomHorizontalFlip** (espelha com probabilidade 0,5).

**Hipótese.** Com mais diversidade de dados, (a) o overfitting deve cair, (b) redes mais profundas podem voltar a compensar e (c) a necessidade de dropout pode diminuir.

**Base:** o campeão de `cnn_b3b_pooling`, com mais épocas e paciência, pois augmentation retarda a convergência. A receita
campeã **sem** augmentation é re-treinada no mesmo bloco e sempre completa os 5 folds, como
referência pareada.

| Eixo | Valores |
|---|---|
| `augment` | `False`, `True` |
| `conv_blocks` | `3`, `4` |
| `cnn_dropout` | `0.3`, `0.5` |

**8 combinações.** Herdado do campeão de: `cnn_b3b_pooling`. Fixos no bloco: `epochs=100`, `patience=10`. Triagem nos folds [1, 2, 3] de 5; as 2 melhores completam os 5 folds. Limite: 20,000,000 parâmetros.

**O que observar:** o ganho pareado sobre a referência sem augmentation, a queda do `gap/accuracy` e se a
`melhor_epoca_media` ficou perto do limite de épocas (resultado possivelmente limitado pelo orçamento).

In [ ]:
!python src/grid_search.py grids/cnn_b4_augmentation.json --workers_per_gpu $WORKERS_PER_GPU

In [ ]:
rep.grid_ranking("cnn_b4_augmentation", "triagem")

In [ ]:
rep.heatmap("cnn_b4_augmentation", row="conv_blocks", col="cnn_dropout", facet="augment")

In [ ]:
rep.heatmap("cnn_b4_augmentation", row="conv_blocks", col="cnn_dropout", facet="augment", value="gap/accuracy_mean")

In [ ]:
rep.heatmap("cnn_b4_augmentation", row="conv_blocks", col="cnn_dropout", facet="augment", value="melhor_epoca_media")

In [ ]:
rep.grid_ranking("cnn_b4_augmentation", "final")

In [ ]:
rep.show_champion("cnn_b4_augmentation")
rep.plot_finalists("cnn_b4_augmentation")

#### Comparação pareada por fold (validação)

Referência: a receita campeã sem augmentation, treinada neste mesmo bloco.

In [ ]:
rep.paired_comparison(rep.base_config_name("cnn_b4_augmentation"), "cnn_b4_augmentation__*")

### 📝 Análise — Bônus

- **Ganho do augmentation na validação (pareado):** _…_
- **O gap treino-validação caiu?** _…_
- **Mudou a configuração ideal (profundidade/dropout)?** _…_
- **Algum resultado ficou limitado pelo orçamento de épocas?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Resultado final — teste revelado

Até aqui todas as escolhas foram feitas pela validação cruzada. Agora o conjunto de teste é usado **uma única vez**
para os campeões de cada bloco e para as referências do Bloco 0. A tabela mostra média ± desvio do teste entre os
5 modelos (um por fold) de cada configuração. O campeão da sequência principal é o de `cnn_b3b_pooling`; o bônus mostra
quanto o augmentation acrescenta sobre ele.

In [ ]:
final = rep.final_report(['cnn_b0_referencia', 'cnn_b1_topologia', 'cnn_b2_otimizacao', 'cnn_b2_checagem', 'cnn_b3_regularizacao', 'cnn_b3b_pooling', 'cnn_b4_augmentation'])
final

Desempenho por classe no teste: referências do Bloco 0 × campeão principal × campeão com augmentation. Compare com as classes mais difíceis das MLPs: as convoluções resolveram as mesmas confusões?

In [ ]:
campeao_principal = rep.champion_name("cnn_b3b_pooling")
campeao_bonus = rep.champion_name("cnn_b4_augmentation")
comparar = list(final.loc[final["papel"] == "referência", "exp_name"]) + [campeao_principal, campeao_bonus]
rep.plot_per_class(comparar, metric="recall")
rep.plot_per_class(comparar, metric="precision")
pd.read_csv(os.path.join(os.environ["EXP_OUTPUT_DIR"], campeao_principal, "matriz_confusao_teste.csv"), index_col=0)

#### Métricas gerais e por classe dos campeões

Para cada campeão: **acurácia, precision, recall e F1 gerais** (macro, no topo da figura) e **precision, recall e F1 de cada classe**
(tabela), no teste, em média ± desvio entre os 5 modelos (um por fold). Por classe, a acurácia é a fração das imagens daquela
classe classificadas corretamente, igual ao recall. Campeões do Bloco 3 (regularização), do Complementar B (campeão principal da CNN) e do Bônus (augmentation).
Figura e CSV: `outputs/_relatorio/metricas_por_classe_test_<experimento>.*`.

In [ ]:
rep.champion_class_metrics("cnn_b3_regularizacao")

In [ ]:
rep.champion_class_metrics("cnn_b3b_pooling")

In [ ]:
rep.champion_class_metrics("cnn_b4_augmentation")

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## 📝 Conclusões

- **Estrutura base que melhor absorveu o CIFAR-10 (Bloco 1) e por quê:** _…_
- **Ganho de otimização (Bloco 2) e sensibilidade à taxa de aprendizagem:** _…_
- **A checagem confirmou a escolha em blocos?** _…_
- **Efeito da regularização/erro (Bloco 3):** _…_
- **O que o experimento complementar corrigiu ou confirmou:** _…_
- **Ganho total sobre a referência (teste):** _…_
- **Bônus — quanto o augmentation acrescentou e por quê:** _…_
- **Classes mais difíceis e hipótese:** _…_